In [2]:
import torch    
import torch_geometric.transforms as T
import mdtraj as md
import os
import torch 
import numpy as np
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data, Dataset
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import BatchNorm1d
from torch_geometric.nn import GATConv, global_mean_pool, GCNConv, knn_graph, Linear
from torch.optim import SGD, Adam, Optimizer
import math
from torch.nn.init import kaiming_uniform_
from torch_geometric.transforms import ToDevice
import scipy
import sys
from e3nn.io import CartesianTensor
from torch_geometric.loader import DataLoader

device='cpu'
import torch
torch.cuda.is_available()

# print(torch.version.cuda)
torch.norm(torch.tensor([-13.36, 2.16, -9.72]))


tensor(16.6623)

In [32]:
df4.to_csv('../../full_contacts4_inr100.csv', index=False)

In [6]:
import pandas as pd

import re

df2 = pd.read_csv('../../full_contacts4.csv')
# mat = re.compile("P4")
df3 = df2[(~df2.PATH.isna()) & (df2.PATH != 'PATH')]
df3.PATH[df3.PATH.str.match(".*P4.*")] = 'P4'
df3['PATH2'] = df3['PATH'].str.split(' ').str[0]
df3.to_csv('../../full_contacts5.csv', index=False)

/tmp/ipykernel_1769802/949824485.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3.PATH[df3.PATH.str.match(".*P4.*")] = 'P4'


In [23]:
df4 = df3[(df3.in_row) & (df3.molecules==100)  & (df3.prop=='prop')]
df4['start'] = df4.range.apply(lambda x: int(x.split('-')[0]))
df4[['FF','start','PATH2']]

/tmp/ipykernel_374820/3284970211.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['start'] = df4.range.apply(lambda x: int(x.split('-')[0]))


,FF,start,PATH2
12,O2QD,96004,P_main
16,O2QD,96611,P_reverse
20,O2QD,95929,P_reverse
24,O2IF,47500,P_main
28,O2IF,56065,P1
...,...,...,...
2544,O2IF,7308,P_main
2548,O2IF,87242,P_main
2552,O2QD,53322,P_main
2556,O2QD,73251,P4


In [24]:
bins = [0, 20000, 30000, 50000, float('inf')]
labels = ["<20000", "20000–30000", "30000–50000", ">50000"]

# Bin the start values
df4["start_bin"] = pd.cut(df4["start"], bins=bins, labels=labels, right=False)

# --- Compute proportions ---
grouped = (
    df4.groupby(["PATH2", "FF"])["start_bin"]
      .value_counts(normalize=True)
      .unstack(fill_value=0)
)

# List of FF values (should be: O2QD, O2IF)
ffs = grouped.index.get_level_values("FF").unique()
grouped

/tmp/ipykernel_374820/4181348911.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4["start_bin"] = pd.cut(df4["start"], bins=bins, labels=labels, right=False)


start_bin         <20000  20000–30000  30000–50000    >50000
PATH2     FF                                                
P1        O2IF  0.000000     0.000000     0.000000  1.000000
          O2QD  0.000000     0.000000     0.222222  0.777778
P2        O2IF  0.000000     0.000000     0.166667  0.833333
          O2QD  0.055556     0.277778     0.222222  0.444444
P3        O2IF  0.181818     0.000000     0.090909  0.727273
          O2QD  0.000000     0.066667     0.400000  0.533333
P4        O2IF  0.018182     0.000000     0.090909  0.890909
          O2QD  0.042553     0.148936     0.276596  0.531915
P_main    O2IF  0.227273     0.106061     0.121212  0.545455
          O2QD  0.082278     0.183544     0.215190  0.518987
P_reverse O2IF  0.010204     0.040816     0.081633  0.867347
          O2QD  0.000000     0.058824     0.235294  0.705882

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# Bin setup
bins = [0, 20000, 30000, 50000, float('inf')]
labels = ["<20000", "20000–30000", "30000–50000", ">50000"]

# Bin the start values
df4["start_bin"] = pd.cut(df4["start"], bins=bins, labels=labels, right=False)

# --- Compute proportions ---
props = (
    df4.groupby(["PATH2", "FF"])["start_bin"]
       .value_counts(normalize=True)
       .unstack(fill_value=0)
)

# --- Compute counts ---
counts = (
    df4.groupby(["PATH2", "FF"])["start_bin"]
       .value_counts()
       .unstack(fill_value=0)
)

# FF values (O2QD, O2IF)
ffs = props.index.get_level_values("FF").unique()

# --- Heatmaps ---
for ff in ffs:
    sub_p = props.loc[(slice(None), ff), :].droplevel("FF")
    sub_c = counts.loc[(slice(None), ff), :].droplevel("FF")

    plt.figure(figsize=(7, 4))
    plt.imshow(sub_p.values, aspect="auto")
    plt.xticks(range(len(sub_p.columns)), sub_p.columns)
    plt.yticks(range(len(sub_p.index)), sub_p.index)
    plt.title(f"Heatmap for FF = {ff}")
    plt.colorbar(label="Proportion")

    # ---- ADD "count (prop)" ----
    for i in range(sub_p.shape[0]):
        for j in range(sub_p.shape[1]):
            count = sub_c.values[i, j]
            prop = sub_p.values[i, j]
            plt.text(
                j, i, f"{count} ({prop:.2f})",
                ha="center", va="center", fontsize=9, color="white"
            )

    plt.tight_layout()
    plt.show()


NameError: name 'df4' is not defined

In [6]:
box=101.7*101.7*107.6
membrane=101.7*101.7*39
1-(membrane/box)

0.6375464684014871

In [3]:
from rdkit import Chem
mol = Chem.MolFromPDBFile("an1.pdb", removeHs=False)
Chem.SanitizeMol(mol)

top ='an1.gro'
Sim=6
t = "../Sim%d/cat_small.dcd"%Sim
traj = md.load(t, top=top)

xyz = torch.tensor(traj.xyz) * 10
ele2num = {"C": 0, "H": 1, "O": 2, "N": 3, "S": 4, "VS": {"FE": 5, "MG":6}} # all 0's will be padding for gases not within 3.5 angstroms of any protein atom
gas='O2QD'
metal='FE'
gas2 = traj.topology.select('resname %s' % gas)
residue_ref = np.array([traj.topology.atom(ind).residue.resSeq for ind in gas2])
FE = traj.topology.select('resname Fe2p')
residue_sel_un = np.unique(residue_ref) # gas
residue_sel_un

print('LEN FE', FE)
residue_sel_un = np.unique(residue_ref) # gas
residue_sel_un
nogas = np.setdiff1d(range(0,traj.xyz.shape[1]),gas2)
rnames = np.array([traj.topology.atom(ind).residue.name for ind in nogas])
rindex = np.array([traj.topology.atom(ind).residue.resSeq for ind in nogas])
anames = np.array([traj.topology.atom(ind).element.symbol for ind in nogas])
anames2=np.array([traj.topology.atom(i).name for i in nogas])
anums = [ele2num[a] if a != 'VS' else ele2num[a][metal] for a in anames]

rnames = np.array([traj.topology.atom(ind).residue.name for ind in nogas])
rindex = np.array([traj.topology.atom(ind).residue.resSeq for ind in nogas])
anames = np.array([traj.topology.atom(ind).element.symbol for ind in nogas])
rnames2 = np.array([traj.topology.atom(ind).residue for ind in nogas])

cat = np.where(anames=='VS')[0]
cat

device = torch.device("cpu")
# device = torch.device("cpu")
protein_coords_traj = xyz[:,nogas,:]
rs = int(len(residue_ref)/len(residue_sel_un))
gas_atoms=gas2.reshape(len(residue_sel_un),3)

dioxygen_coords_ave = xyz[:,gas2,:].reshape(-1,len(residue_sel_un),rs,3).mean(axis=2)
d=torch.cdist(dioxygen_coords_ave, xyz[:,cat,:])
diox = np.where(d < 6.0)[1]
frames = np.where(d < 6.0)[0]

types_array_atom = torch.zeros((len(nogas)+len(gas2), (len(ele2num))))
for i, t in enumerate(anums):
    types_array_atom[i,t] = 1.0
# types_array_atom.shape
types_array_atom[-len(gas2):,ele2num['O']] = 1
types_array_atom = types_array_atom.to(device)

gas_idx = np.where(residue_sel_un==250000)[0]
gas_idx
protein_cofactors = torch.tensor(nogas).to(device)
global_to_local = {int(g): i for i, g in enumerate(np.arange(0,xyz[:,nogas,:].shape[1]))}

del traj

t = "../Sim16/cat_small.dcd"
traj = md.load_frame(t, index=0,top=top )

/home/coyote/miniconda3/envs/holoprot/lib/python3.10/site-packages/mdtraj/formats/gro.py:364: UserWarning: WARNING: two consecutive residues with same number (GLN, ACE)
  warnings.warn(
/home/coyote/miniconda3/envs/holoprot/lib/python3.10/site-packages/mdtraj/formats/gro.py:364: UserWarning: WARNING: two consecutive residues with same number (ASP, ACE)
  warnings.warn(


LEN FE [3790]


In [ ]:
close = traj.topology.select('resname VAL and residue 314 and name O')
close
d2=torch.cdist(dioxygen_coords_ave, xyz[:,close,:])

In [12]:
d2.min()

tensor(2.8227)

In [17]:
protein_cofactors

tensor([   0,    1,    2,  ..., 3788, 3789, 3790])

In [10]:
# types_array_atom.shape
from torch_geometric.data import Data
def extract_point_cloud(atom_matrix, positions, center):
    """
    Formats the already-cropped atom features and positions into a Data object.
    """
    if positions.shape[0] == 0:
        return None  # Skip empty cubes

    # Center coordinates relative to cube center
    centered_positions = positions - center

    return Data(x=atom_matrix, pos=centered_positions)

path_dict = {
    'P1':'P1', 
    'P_main (PmR)':'PmR', 
    'P_main (mid)':'mid', 
    'P_reverse':'P_reverse', 
    'P_main (PmL)':'PmL',
       'P3':'P3'
}

def write_pdb2(inds, what, xs, ys, zs, chain='C', file="ml_out.pdb"):
    
    print(file)
    
    fpdb = open(file, 'wt')
    norm = torch.max(what[inds])
    i_atom = 1
    i_resid = 1
    for i, dind in enumerate(inds):

        fpdb.write('{:6s}{:5d} {:^4s}{:1s}{:3s} {:1s}{:4d}{:1s}   {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}          {:>2s}{:2s}\n'.format(
            'ATOM',i_atom,
            'GG','','GG',
            chain,i_resid,'',
            xs[dind],ys[dind],zs[dind],
            -1.0*torch.log10(what[dind]/norm),(what[dind]),
            'K',''))
        i_atom += 1
        if i_atom > 999:
            i_atom = 1
            i_resid += 1
            #eigv[dind,0],mm.pi[dind],
    fpdb.write('TER\n')
    fpdb.close()

def get_points(start, translated, frame_pos, space=0.65, bottom_threshold=2.5):
    x_edges = np.arange(-20, 21, space)
    y_edges = np.arange(-20, 21, space)
    z_edges = np.arange(-20, 21, space)

    X, Y, Z = np.meshgrid(x_edges, y_edges, z_edges, indexing='ij')

    # Stack the arrays to create a 3D array of shape (N, 3)
    points_3d = torch.tensor(np.stack([X, Y, Z], axis=-1).reshape(-1, 3), dtype=torch.float32)

    translated_ang = translated.clone()
    exclude_points = translated_ang

    threshold = bottom_threshold
    threshold2 = 3.5

    # Compute distances from every point in `points_3d` to every `exclude_point`
    distances=torch.cdist(points_3d, exclude_points)

    # Find points in `points_3d` with all distances >= threshold
    mask = torch.all(distances >= threshold, axis=1)
    mask2 = torch.any(distances < threshold2, axis=1)

    # Filter `points_3d` to keep only points outside the threshold
    filtered_points = points_3d[mask & mask2]

    filtered_points2 = filtered_points# + protein_coords_list.cpu()[cat].numpy() 
    filtered_points2.mean(axis=0)
    filtered_points2.shape
    dist2 = torch.norm(filtered_points2-frame_pos, dim=1)
    if start==0:
        filtered_points3 = filtered_points2[dist2 <= 20]
    else:
        filtered_points3 = filtered_points2[dist2 <= 5]
    xyz2=filtered_points3

    return xyz2

def get_fe_bias(xyz2, frame_pos, alpha=10.0, beta=1.0, denom=4, range_=0.75):
    """
    alpha → dominance of origin distance
    beta  → influence of closeness to the line
    """

    # ------------------------------
    # Distance to origin (dominant term)
    # ------------------------------
    dist_origin = torch.norm(xyz2, dim=1)  # [N]
    # Invert + square for extreme emphasis (close gets very large)
    origin_score = 1.0 / (dist_origin**denom + 1e-8)

    # ------------------------------
    # Distance to line
    # ------------------------------
    line_dir = frame_pos / torch.norm(frame_pos)
    t = (xyz2 * line_dir).sum(dim=1, keepdim=True)
    d_perp = torch.norm(xyz2 - t * line_dir, dim=1)

    # Invert perpendicular distance
    line_score = 1.0 / (d_perp + 1e-8)

    # ------------------------------
    # Combine WITHOUT global normalization
    # (so origin dominance is preserved)
    # ------------------------------
    combined = alpha * origin_score + beta * line_score

    # ------------------------------
    # Scale result to 0–0.5 while preserving ranking
    # ------------------------------
    combined_min = combined.min()
    combined_max = combined.max()

    score = range_ * (combined - combined_min) / (combined_max - combined_min + 1e-8)

    return score

import torch
def get_best(density, pos_embedding, k=5):
# points: [N,3], density: [N]
    # k = 5
    points = torch.stack([p.center for p in pos_embedding])
    # density=density
    N = points.shape[0]

    best_sum = -1
    best_group = None
    best_ndx = None
    scores = {}

    for i in range(N):
        # compute distances from point i
        dists = torch.norm(points - points[i], dim=1)  # [N]
        
        # find indices of the closest k points including self
        _, nn_idx = torch.topk(-dists, k)  # negative because topk returns largest
        
        # sum densities in this group
        group_density = density[nn_idx].sum()
        scores[i] = {}
        scores[i]['group_density'] = group_density
        scores[i]['nn_idx'] = nn_idx
        
        if group_density > best_sum:
            best_sum = group_density
            best_group = nn_idx
            best_ndx = i

    highest_cluster_points = points[best_group]
    highest_cluster_density = density[best_group]

    return scores, points


def embed(xyz2, step, protein_coords_list,protein_cofactors,global_to_local, radius=5, frame_emb=None):
    print("STEP:", step)
    # xyz2:  grid point coordinates for the frame we are trying to predictt, i.e. the next frame, 
    # frame_pos: the given O2 in frame 0
    # first frame frame_pos for O2 is given, each one after is prediction from one before
    # 0 is first step which is given and we start to predict from 1

    pos_embedding = []
    xyz = []
    atom_coords = protein_coords_list
    
    # print(dist)
    min_dist = 2.5
    
    for i, point in enumerate(xyz2[0:]):
        center = point
        dist = torch.norm(center - atom_coords[cat[0],:])
      
      
        # radius = 6.0

        # Compute distances of all atoms to perturbed point
        dists = torch.norm(atom_coords - center, dim=1)

        # Indices of atoms inside sphere
                    # Indices of atoms inside sphere
        inside_indices = protein_cofactors[torch.where(dists <= radius)[0]]

        # Ensure inside_indices is always a 1D tensor
        if torch.is_tensor(inside_indices) and inside_indices.ndim == 0:
            inside_indices = inside_indices.unsqueeze(0)

        if isinstance(inside_indices, (int, np.integer)):
            inside_indices = torch.tensor([inside_indices], device=device)

        if inside_indices.numel() == 0:
            dddd = torch.cdist(center.unsqueeze(0), atom_coords)
            print('too far start or end', dddd.min(), flush=True)
            continue
        xyz.append(i)
        local_inside_indices = np.array([global_to_local[int(g)] for g in inside_indices])

        atom_matrix=types_array_atom[inside_indices]
        positions=atom_coords[local_inside_indices]
        node_directions = positions - atom_coords[cat[0],:]
        node_distances = torch.norm(node_directions, dim=1)
        local_pos_normalized = (positions - center)/radius  # shape [N, 3]
        
        # atom_matrix_arr.append(atom_matrix)
        # positions_arr.append(positions)
        test = extract_point_cloud(atom_matrix, positions, center)
        # test.x=torch.column_stack([test.x, torch.tensor(frame_emb[idx]).repeat(len(test.x))])
        if frame_emb is not None:
            frame_vector = frame_emb[step]          # [16]
            frame_vector = frame_vector.unsqueeze(0)           # [1, 16]
            frame_vector = frame_vector.expand(len(test.x), -1)  # [N, 16]
            test.distance = dist.expand(len(test.x))
            test.x = torch.cat([test.x, frame_vector], dim=1)    # [N, 6 + 16]

        test.edge_index, test.edge_attr = build_edges_and_attrs(inside_indices)
        test.inside_indices = inside_indices
        test.center = center
        test.node_attr = local_pos_normalized
        test.node_directions = node_directions
        test.node_distances = node_distances
        pos_embedding.append(test)

    xyz2 = xyz2[xyz]
    return xyz2, pos_embedding

def topk_with_radius(density, pos_embedding, k=5, radius=2.0):

    points = torch.stack([p.center for p in pos_embedding])
    N = points.shape[0]
    best_sum = -1
    best_group = None
    best_ndx = None
    scores = {}

    for i in range(N):
        # distances from point i
        dists = torch.norm(points - points[i], dim=1)

        # initial K nearest indices (including itself)
        _, nn_idx = torch.topk(-dists, k)

        # extract positions of candidate group
        group_pts = points[nn_idx]            # [k,3]

        # compute pairwise distance matrix within group
        pdist = torch.norm(
            group_pts.unsqueeze(1) - group_pts.unsqueeze(0),
            dim=2
        )  # [k, k]
        # check radius requirement: all pairwise distances ≤ radius
        if (pdist <= radius).all():
            group_density = density[nn_idx].sum()

            if group_density > best_sum:
                best_sum = group_density
                best_group = nn_idx
                best_ndx = i
            scores[i] = {}
            scores[i]['group_density'] = group_density
            scores[i]['nn_idx'] = nn_idx

    return scores, points

# Map from global RDKit atom idx → local 0..N-1
import itertools

def build_edges_and_attrs(inside_indices):
    atom_to_local = {int(a): i for i, a in enumerate(inside_indices.tolist())}

        # --- Extract bonded edges from RDKit ---
    bonded_edges = []
    bonded_attrs = []

    for bond in mol.GetBonds():
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()

        # Only keep if both atoms are in the subset
        if a1 in atom_to_local and a2 in atom_to_local and a1 not in gas_atoms and a2 not in gas_atoms:
            i1, i2 = atom_to_local[a1], atom_to_local[a2]

            bt = bond.GetBondType()
            if bt == Chem.rdchem.BondType.SINGLE:
                order = 1
            elif bt == Chem.rdchem.BondType.DOUBLE:
                order = 2
            elif bt == Chem.rdchem.BondType.AROMATIC:
                order = 3
            else:
                order = 1

            # undirected edges
            bonded_edges += [[i1, i2], [i2, i1]]
            bonded_attrs += [order, order]

    for (a1, anot ,a2) in gas_atoms:
        if a1 in atom_to_local and a2 in atom_to_local:
            i1, i2 = atom_to_local[a1], atom_to_local[a2]
            bonded_edges += [[i1, i2], [i2, i1]]
            bonded_attrs += [1, 1]

    # --- Build nonbonded edges among all pairs in subset ---
    N = len(inside_indices)
    all_pairs = list(itertools.combinations(range(N), 2))

    bonded_set = set(tuple(sorted(e)) for e in [(a, b) for a, b in bonded_edges if a < b])

    nonbonded_edges = []
    nonbonded_attrs = []

    for i, j in all_pairs:
        if (i, j) not in bonded_set:
            nonbonded_edges += [[i, j], [j, i]]
            nonbonded_attrs += [0, 0]  # edge_attr = 0 for nonbonded

    # --- Combine everything ---
    # edge_index = torch.tensor(bonded_edges + nonbonded_edges, dtype=torch.long).T
    edges = bonded_edges + nonbonded_edges  # list of [i, j] pairs
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(bonded_attrs + nonbonded_attrs, dtype=torch.long)

    return edge_index, edge_attr.unsqueeze(1)

import torch
import math

def predict(pos_embedding, model, bs=5, frame_emb_bool=False, node_input_bool=False, atom_as_node_attr=False):
    # model = torch.load('../Sim1/model.pt') # 
    # model=torch.load("../Sim5/model%d_FE2dis_inr6.pt"%24)
    model=torch.load(model)
    device='cuda'
    model = model.to(device)


    test_loader = DataLoader(pos_embedding, batch_size=bs, shuffle=False) 


    values = []
    with torch.no_grad():
        for batch_idx, (data_list) in enumerate(test_loader):
            if (batch_idx + 1) % 20 == 0:
                print("Batch",batch_idx+1, flush=True)
            if isinstance(data_list, list):
                from torch_geometric.data import Batch
                batch = Batch.from_data_list(data_list)
            else:
                batch = data_list  # if batch_size=1, it might already be a Data object

            batch = batch.to('cuda')
            

            
            distance = torch.norm(batch.node_attr[:, :3], dim=1, keepdim=True)  # [N, 1]

            # 2. Convert Cartesian vectors to irreps vector
            x = CartesianTensor("i")
            vector_irrep = x.from_cartesian(batch.node_attr[:, :3])  # [N, 3]

          
            atom_type_onehot = batch.x[:, 0:6]
            
            frame_emb = batch.x[:, 6:] 
            if frame_emb_bool:
                if atom_as_node_attr:
                    node_attr = torch.cat([
                        distance,            # 1 scalar (0e)
                        atom_type_onehot,    # 6 scalars (0e)
                        frame_emb,           # k scalars (0e)
                        vector_irrep,        # 3-vector (1o)
                    ], dim=1)
                else:
                    node_attr = torch.cat([
                        distance,            # 1 scalar (0e)
                        # atom_type_onehot,    # 6 scalars (0e)
                        frame_emb,           # k scalars (0e)
                        vector_irrep,        # 3-vector (1o)
                    ], dim=1)

            else:
                # node_attr = torch.cat([distance, vector_irrep, frame_emb], dim=1)
                if atom_as_node_attr:
                    node_attr = torch.cat([
                        distance,            # 1 scalar (0e)
                        atom_type_onehot,    # 6 scalars (0e)
                        # frame_emb,           # k scalars (0e)
                        vector_irrep,        # 3-vector (1o)
                    ], dim=1)
                else:
                    node_attr = torch.cat([
                        distance,            # 1 scalar (0e)
                        # atom_type_onehot,    # 6 scalars (0e)
                        # frame_emb,           # k scalars (0e)
                        vector_irrep,        # 3-vector (1o)
                    ], dim=1)

            
            # node_input = torch.ones((batch.num_nodes, 1), device=batch.x.device)
            if node_input_bool:
                node_input = torch.cat([
                    # batch.distance.unsqueeze(-1), # 1 scalar (0e) distance of center of graph to iron
                    batch.node_distances.unsqueeze(-1),    # 1 scalar (0e) distance to iron for each atom
                    F.normalize(batch.node_directions, p=2, dim=1),        # 3-vector (1o) direction to iron for each atom
                ], dim=1)
            else:
                node_input = torch.ones((batch.num_nodes, 1), device=batch.x.device)

            if not atom_as_node_attr:
                data = {
                "batch": batch.batch,
                # "x": batch.x[:,0:6], # atom type
                # "frame_emb": frame,
                "x": atom_type_onehot,
                "node_attr": node_attr, 
                "edge_index": batch.edge_index,
                "edge_attr": batch.edge_attr,
                "pos": batch.pos,  # if needed in preprocess
            }
            else:
                data = {
                    "batch": batch.batch,
                    # "x": batch.x[:,0:6], # atom type
                    # "frame_emb": frame,
                    "x": node_input,
                    "node_attr": node_attr, 
                    "edge_index": batch.edge_index,
                    "edge_attr": batch.edge_attr,
                    "pos": batch.pos,  # if needed in preprocess
                }

            outputs = model(data)
            preds = torch.sigmoid(outputs.squeeze(-1))
            values.append(preds)

        results = torch.concat(values)
        best = results.argmax() # 2552
        best_pos = best#xyz2[best]
        return best, best_pos, results

def sinusoidal_embedding(frame_idx: torch.Tensor, dim: int = 16):
    """
    frame_idx: tensor of shape [N] with normalized frame values in [0,1]
    dim: embedding dimension (should be even)
    Returns: tensor of shape [N, dim]
    """
    device = frame_idx.device
    N = frame_idx.size(0)
    pe = torch.zeros(N, dim, device=device)

    position = frame_idx.unsqueeze(1)  # [N, 1]
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * -(math.log(10000.0) / dim))  # [dim/2]

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # [N, dim]

    

In [16]:
model.irreps_node_attr

1x0e+1x1o

In [11]:
start=0
end=100
frames = torch.arange(0, end, dtype=torch.float32)  # [0,1,...,N-1]
frame_norm = frames / (end - start)    
frame_emb = sinusoidal_embedding(frame_norm, dim=16)

In [13]:
frame_emb

tensor([[0.0000e+00, 1.0000e+00, 0.0000e+00,  ..., 1.0000e+00, 0.0000e+00,
         1.0000e+00],
        [9.9998e-03, 9.9995e-01, 3.1623e-03,  ..., 1.0000e+00, 3.1623e-06,
         1.0000e+00],
        [1.9999e-02, 9.9980e-01, 6.3245e-03,  ..., 1.0000e+00, 6.3246e-06,
         1.0000e+00],
        ...,
        [8.2489e-01, 5.6530e-01, 3.0195e-01,  ..., 1.0000e+00, 3.0674e-04,
         1.0000e+00],
        [8.3050e-01, 5.5702e-01, 3.0497e-01,  ..., 1.0000e+00, 3.0990e-04,
         1.0000e+00],
        [8.3603e-01, 5.4869e-01, 3.0798e-01,  ..., 1.0000e+00, 3.1307e-04,
         1.0000e+00]])

In [5]:
model = torch.load('../Sim5/models/output_ep_20_bs_12_lr_0.0003_opt_adamw_inw_kaiming_neigh_20_nodes_65_mul_40_lay_2_lmax_2.pt')

In [12]:
# end=len(traj)
# frames = torch.arange(0, end, dtype=torch.float32)  # [0,1,...,N-1]
# frame_norm = frames / (end - start)    
# frame_emb = sinusoidal_embedding(frame_norm, dim=16) 
s=1

frame_pos=torch.tensor([0,0,0])
start=0
# start=15000
# start=8
step=0
# frame_pos=torch.tensor([4.75, 12.5, 2.0])
# for start in [21000, 31000, 0, 1000, 5000, 10000, 19000, 19900]:
for start in [500]:
# for start in [19900]:
    print("START:", start)
    protein_coords_traj = xyz[start,nogas,:]
    protein_coords_list = protein_coords_traj
    translated = protein_coords_list - protein_coords_list.cpu()[cat].numpy() 
    new_traj = md.Trajectory(xyz=((xyz[start]-protein_coords_traj.cpu()[cat].numpy())/10).numpy(), topology=traj.topology)
    new_traj.save('translated_frame%d_Sim%d.pdb'%(start,Sim))

    x_edges = np.arange(-20, 21, s)
    y_edges = np.arange(-20, 21, s)
    z_edges = np.arange(-20, 21, s)

    X, Y, Z = np.meshgrid(x_edges, y_edges, z_edges, indexing='ij')

    # Stack the arrays to create a 3D array of shape (N, 3)
    points_3d = torch.tensor(np.stack([X, Y, Z], axis=-1).reshape(-1, 3), dtype=torch.float32)

    translated_ang = translated.clone()
    exclude_points = translated_ang


    threshold = 2.5
    threshold2 = 4.5

    # Compute distances from every point in `points_3d` to every `exclude_point`
    distances=torch.cdist(points_3d, exclude_points)

    # Find points in `points_3d` with all distances >= threshold
    mask = torch.all(distances >= threshold, axis=1)
    mask2 = torch.any(distances < threshold2, axis=1)

    # Filter `points_3d` to keep only points outside the threshold
    filtered_points = points_3d[mask & mask2]

    filtered_points2 = filtered_points# + protein_coords_list.cpu()[cat].numpy() 
    filtered_points2.mean(axis=0)
    filtered_points2.shape
    dist2 = torch.norm(filtered_points2-frame_pos, dim=1)
    # if start%100 == 0 or start==19999:
    filtered_points3 = filtered_points2[(dist2 <= 25) & (dist2 >= 12)]
    # else:
    #     filtered_points3 = filtered_points2[dist2 <= 6]
    xyz2=filtered_points3
    print(xyz2.shape)
    # # embed_predict and get highest next point to be the next frame_pos at 2...end
    xyz2, pos_embedding = embed(xyz2, step=step, protein_coords_list=translated_ang,protein_cofactors=protein_cofactors,global_to_local=global_to_local, frame_emb=frame_emb)
    for i, pos in enumerate(pos_embedding):
        if pos.edge_attr.shape[0] == 0:
            pos.edge_index = torch.tensor([[0],[0]])
            pos.edge_attr = torch.tensor([[0]])
    
    torch.save(pos_embedding,"pos_embedding_Sim%d_f%d_1s_2.5-4.5d.pt"%(Sim,start))
    

    # index, frame_pos__, results = predict(pos_embedding, model='../Sim5/model24_FE2dis_inrAll_far2.pt', bs=5) ## using beginning of entry to 10 ang; all sims 100 mols pos_embedding_in_far.pt and neg_embedding_in_far.pt; log at /data/pompei/bw973/Oxygenases/PHD2/PHD2_O2QD/Bundle/Sim5/FE2dis_inrAll_far.log
    torch.cuda.empty_cache()
    # index, frame_pos__, results2 = predict(pos_embedding, model='../../../PCOs/O2IF/Sim5/model15_FE2dis_inr5.pt', bs=5) ## log at /media/bw973/Seagate Hub/Oxygenases/PCO_MUTANTS/ARABI/PCO4_WT_100_O2IF/Sim5/model.log; Sims [1,3,4,5,6,7,8,9,10]
    # torch.cuda.empty_cache()
    index, frame_pos__, results3 = predict(pos_embedding, model='../Sim5/models/output_ep_20_bs_12_lr_0.0003_opt_adamw_inw_kaiming_neigh_20_nodes_65_mul_40_lay_2_lmax_2.pt', bs=5)
    torch.cuda.empty_cache()
    write_pdb2(torch.arange(0,len(filtered_points3)),results3,filtered_points3[:,0],filtered_points3[:,1],filtered_points3[:,2], file=str(start)+ '_frame' + str(step) + '_of_' + str((end)) + 'far2_Sim%d.pdb'%Sim)

    myd = {}
    # myd['model24_FE2dis_inrAll_far2.pt'] = results
    # myd['PCOs_model15_FE2dis_inr5.pt'] = results2
    myd['model30_FE2dis_inr7.pt'] = results3
    torch.save(myd, "%d_start_test_Sim6.dict" % start)


START: 500
torch.Size([9275, 3])
STEP: 0
Batch 20
Batch 40
Batch 60
Batch 80
Batch 100
Batch 120
Batch 140
Batch 160
Batch 180
Batch 200
Batch 220
Batch 240
Batch 260
Batch 280
Batch 300
Batch 320
Batch 340
Batch 360
Batch 380
Batch 400
Batch 420
Batch 440
Batch 460
Batch 480
Batch 500
Batch 520
Batch 540
Batch 560
Batch 580
Batch 600
Batch 620
Batch 640
Batch 660
Batch 680
Batch 700
Batch 720
Batch 740
Batch 760
Batch 780
Batch 800
Batch 820
Batch 840
Batch 860
Batch 880
Batch 900
Batch 920
Batch 940
Batch 960
Batch 980
Batch 1000
Batch 1020
Batch 1040
Batch 1060
Batch 1080
Batch 1100
Batch 1120
Batch 1140
Batch 1160
Batch 1180
Batch 1200
Batch 1220
Batch 1240
Batch 1260
Batch 1280
Batch 1300
Batch 1320
Batch 1340
Batch 1360
Batch 1380
Batch 1400
Batch 1420
Batch 1440
Batch 1460
Batch 1480
Batch 1500
Batch 1520
Batch 1540
Batch 1560
Batch 1580
Batch 1600
Batch 1620
Batch 1640
Batch 1660
Batch 1680
Batch 1700
Batch 1720
Batch 1740
Batch 1760
Batch 1780
Batch 1800
Batch 1820
Batch 1840


In [4]:
Sim=6
start=500
myd=torch.load("%d_start_test_Sim6.dict" % start)
results3 = myd['model30_FE2dis_inr7.pt']
pos_embedding=torch.load("pos_embedding_Sim%d_f%d_1s_2.5-4.5d.pt"%(Sim,start))

In [18]:
import torch

start=6580
myd = torch.load("%d_start.dict" % start)
results=myd['model35_FE2dis_inrAll_far.pt']
results2=myd['PCOs_model15_FE2dis_inr5.pt']
results3=myd['model30_FE2dis_inr7.pt']
results4 = torch.load("test_out_20_mPHD2_f%d.pt"%start)
results.shape

torch.Size([18864])

In [21]:
# index, frame_pos__, results = predict(pos_embedding, model='../Sim5/model25_FE2dis_inrPmP4_far.pt')
# index, frame_pos__, results = predict(pos_embedding, model='../Sim5/model27_FE2dis_inrAll_far.pt')
write_pdb2(torch.arange(0,len(filtered_points3)),results+results2+results3,filtered_points3[:,0],filtered_points3[:,1],filtered_points3[:,2], file=str(start)+ '_frame' + str(step) + '_of_' + str((end)) + 'far2.pdb')

6580_frame0_of_100far2.pdb


In [5]:
def topk_with_radius(density, pos_embedding, k=5, radius=2.0, threshold = 0.5):
    points = torch.stack([p.center for p in pos_embedding])
    
    keep = torch.where(density >= threshold)[0].to('cpu')
    print(points.device, keep.device)
    points=points[keep]
    density = density.to('cpu')[keep]
    N = points.shape[0]
    best_sum = -1
    scores = {}

    for i in range(N):
        # distances from point i
        dists = torch.norm(points - points[i], dim=1)

        # initial K nearest indices (including itself)
        _, nn_idx = torch.topk(-dists, k)

        # extract positions of candidate group
        group_pts = points[nn_idx]            # [k,3]

        # compute pairwise distance matrix within group
        pdist = torch.norm(
            group_pts.unsqueeze(1) - group_pts.unsqueeze(0),
            dim=2
        )  # [k, k]

        # check radius requirement: all pairwise distances ≤ radius
        if (pdist <= radius).all():
            group_density = density[nn_idx].sum()

            if group_density > best_sum:
                best_sum = group_density
            scores[i] = {}
            scores[i]['group_density'] = group_density
            scores[i]['nn_idx'] = nn_idx

    return scores, points

In [8]:
scores, points = topk_with_radius(results3, pos_embedding, k=10, radius=4, threshold=0.3)

cpu cpu


In [7]:
results3.mean()

tensor(0.3994, device='cuda:0')

In [16]:
# print(scores[3806])
# print(scores[3806]['nn_idx'].numpy())
top100

NameError: name 'top100' is not defined

In [ ]:
c: FIXEDATOM AT=0.5915449,0.91589033,26.97987352
# End point of the axis
d: FIXEDATOM AT=-0.47780228,-2.338205,-24.90719

colors = ['blue','green','black','orange','yellow', 'purple', 'red']   
start = np.array([0.5915449,0.91589033,26.97987352])
end = np.array([-0.47780228,-2.338205,-24.90719])


In [ ]:
R322    76.00
I566     73.79
F391    59.45
T296    57.79
W389   43.56
R396    42.49
P564    15.74
D320    12.38



In [9]:
top100 = np.array(sorted(scores.items(), key=lambda x: x[1]['group_density'], reverse=True))[0:100]#[[36, 29, 43, 82]]

for key, vals in top100:
    frame_pos=points[vals['nn_idx']].mean(axis=0)
    # print(vals['group_density'])
    print("draw sphere {",frame_pos.numpy()[0],frame_pos.numpy()[1],frame_pos.numpy()[2],"} radius 1")

draw sphere { -7.5 6.8 19.4 } radius 1
draw sphere { -7.6 7.2 19.4 } radius 1
draw sphere { -8.0 7.4 19.4 } radius 1
draw sphere { -8.0 7.4 19.4 } radius 1
draw sphere { -0.9 -12.5 15.1 } radius 1
draw sphere { -0.9 -12.5 15.1 } radius 1
draw sphere { -8.2 6.9 19.4 } radius 1
draw sphere { -0.9 -12.7 15.0 } radius 1
draw sphere { -0.9 -12.7 15.0 } radius 1
draw sphere { -0.9 -12.7 15.0 } radius 1
draw sphere { -15.8 -0.1 -0.4 } radius 1
draw sphere { -15.8 -0.1 -0.4 } radius 1
draw sphere { -7.3 6.8 19.6 } radius 1
draw sphere { -15.7 0.0 -0.6 } radius 1
draw sphere { -8.0 6.4 19.4 } radius 1
draw sphere { -7.1 6.1 19.6 } radius 1
draw sphere { -16.0 -0.5 -0.2 } radius 1
draw sphere { -16.0 -0.5 -0.4 } radius 1
draw sphere { -6.9 5.8 19.7 } radius 1
draw sphere { -6.9 5.8 19.7 } radius 1
draw sphere { -8.8 7.2 19.4 } radius 1
draw sphere { -7.1 6.1 19.7 } radius 1
draw sphere { -15.9 0.1 -0.6 } radius 1
draw sphere { -15.9 0.1 -0.6 } radius 1
draw sphere { -3.1 16.9 2.8 } radius 1
draw

In [37]:
scores[424]

{'group_density': tensor(10.0409), 'nn_idx': tensor([424, 314, 423, 416, 322])}

In [29]:
import numpy as np

new_start = np.array([0,0,0])  # center of new molecule usually metal
original_start = np.array([79.700691, 79.287521, -7.257360]) # original origin metal

p_start = original_start
# p_end=np.array([60.0055534, 62.88590774, -12.06023983]) # P_main
# p_end=np.array([87.54330488,  55.5394315, -16.25374099]) # P1
# p_end=np.array([69.38878851, 56.73535277, -16.60638655]) # P2
# p_end=np.array([66.19640655, 87.19208037, -26.06862637]) # P3
p_end=np.array([99.41375978, 92.68580929, -4.6561305]) # P_reverse
# p_end=np.array([78.02982585, 74.87044763, -32.16795302]) # P4



#O2QD roto matrix
# Define 4x4 roto-translation matrix

# MSM roto matrix
M = np.array([
    [0.9994531869888306, 0.01978638768196106, 0.026492320001125336, -81.68108367919922],
    [-0.021798361092805862, 0.996719241142273, 0.07794592529535294, -77.58331298828125],
    [-0.024863136932253838, -0.0784807950258255, 0.9966055154800415, 16.090787887573242],
    [0.0, 0.0, 0.0, 1.0]
])

# Convert to homogeneous coordinates (add 1 for matrix multiplication)
p_start_hom = np.append(p_start, 1)  # [x, y, z, 1]
p_end_hom = np.append(p_end, 1)  # [x, y, z, 1]

# Apply the roto-translation
p_start_transformed = M @ p_start_hom
p_end_transformed = M @ p_end_hom

# Extract only the (x, y, z) components
p_start_transformed = p_start_transformed[:3]
p_end_transformed = p_end_transformed[:3]

# Shift the transformed vector to start at (0,0,0)
p_vector_transformed = p_end_transformed - p_start_transformed


# Shift the transformed vector to start at (0,0,0)
p_vector_transformed = p_end_transformed - p_start_transformed

print("New Start Point (should be 0,0,0):", np.array([0,0,0]))
print("New End Point:", p_vector_transformed)
# p_vector_transformed2  = p_vector_transformed + np.array([19.817844,4.080989,-15.953906]) 
# print("vmd_draw_arrow 6 {19.817844 4.080989 -15.953906} {" + str(p_vector_transformed2[0]) + ' ' + str(p_vector_transformed2[1]) + ' ' + str(p_vector_transformed2[2]) + '}')
p_vector_transformed2  = p_vector_transformed + new_start
print("vmd_draw_arrow top {0 0 0} {" + str(p_vector_transformed2[0]) + ' ' + str(p_vector_transformed2[1]) + ' ' + str(p_vector_transformed2[2]) + '}')

New Start Point (should be 0,0,0): [0 0 0]
New End Point: [20.03630575 13.12737439  1.05076262]
vmd_draw_arrow top {0 0 0} {20.036305748192007 13.127374385783504 1.0507626213129058}


In [ ]:
vmd_draw_arrow top {0 0 0} {-11.000167154277776 -22.982115955338227 -7.290993088211859} # p2
vmd_draw_arrow top {0 0 0} {-20.136136150458825 -16.29284671327068 -3.00968197736654} # p_main
vmd_draw_arrow top {0 0 0} {7.130101527096798 -24.542365114861546 -7.29706595249457} # p1
vmd_draw_arrow top {0 0 0} {-13.838851543706753 6.706736122384501 -19.0320090477402} # p3
vmd_draw_arrow top {0 0 0} {-2.4172888270182114 -6.307839118138911 -24.437836018825838} # p4
vmd_draw_arrow top {0 0 0} {20.036305748192007 13.127374385783504 1.0507626213129058} # p_r


vmd_draw_arrow top {0 0 0} {-11.000167154277776 -22.982115955338227 -7.290993088211859}
vmd_draw_arrow top {0 0 0} {-20.136136150458825 -16.29284671327068 -3.00968197736654}
vmd_draw_arrow top {0 0 0} {7.130101527096798 -24.542365114861546 -7.29706595249457}
vmd_draw_arrow top {0 0 0} {-13.838851543706753 6.706736122384501 -19.0320090477402}
vmd_draw_arrow top {0 0 0} {-2.4172888270182114 -6.307839118138911 -24.437836018825838}
vmd_draw_arrow top {0 0 0} {20.036305748192007 13.127374385783504 1.0507626213129058}

In [ ]:
top10 = np.array(sorted(scores.items(), key=lambda x: x[1]['group_density'], reverse=True))[[35, 28, 42, 81]]

for key, vals in top10:
    frame_pos=points[vals['nn_idx']].mean(axis=0)
    # print(key, vals['group_density'], points[vals['nn_idx']].mean(axis=0))
    print("draw sphere {",frame_pos.numpy()[0],frame_pos.numpy()[1],frame_pos.numpy()[2],"} radius 1")

draw sphere { 10.48 -6.2 -4.32 } radius 1
draw sphere { -3.6 -4.24 -18.16 } radius 1
draw sphere { 4.28 -13.36 -7.36 } radius 1
draw sphere { 4.52 -6.64 -12.72 } radius 1
